In [1]:
# ============================================================
# CELL 1: Mount Drive, Fast Local Copy, Imports & Class Weights
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Copy to local disk so DataLoader runs fast during 448px training
!cp -r /content/drive/MyDrive/Week-4 /content/Week-4

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
np.random.seed(42)
print("Device:", DEVICE)

BASE = Path("/content/Week-4")
CKPT_DIR = BASE / "results/checkpoints"
PRED_DIR = BASE / "results/predictions"
METRIC_DIR = BASE / "results/metrics"
for d in [CKPT_DIR, PRED_DIR, METRIC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

splits   = json.loads((BASE / "data/splits.json").read_text())
excluded = set(json.loads((BASE / "data/excluded_ids.json").read_text()))
hard_ids = json.loads((BASE / "data/hard_set.json").read_text())

def get_records(split_name):
    return [
        (k, BASE / f"data/cutouts/{k}.npz", BASE / f"results/pseudo_labels/{k}.npy")
        for k in splits[split_name] if k not in excluded
    ]

class DESISegDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        key, img_p, mask_p = self.records[idx]
        rgb = np.load(img_p)["rgb"].astype(np.float32)
        img = torch.from_numpy(rgb).permute(2, 0, 1)
        mask = torch.from_numpy(np.load(mask_p).astype(np.int64))
        return key, img, mask

train_records = get_records("train")
val_records   = get_records("validation")
test_records  = get_records("test")

pixel_counts = np.zeros(3, dtype=np.int64)
for _, _, mp in train_records:
    m = np.load(mp)
    for c in range(3):
        pixel_counts[c] += (m == c).sum()

freqs = pixel_counts / pixel_counts.sum()
class_weights = torch.tensor(1.0 / np.log(1.02 + freqs), dtype=torch.float32, device=DEVICE)
print(f"Train={len(train_records)}, Val={len(val_records)}, Test={len(test_records)}")
print("Class weights:", np.round(class_weights.cpu().numpy(), 3))

Mounted at /content/drive
Device: cuda
Train=2099, Val=300, Test=600
Class weights: [ 1.474 31.266 18.079]


In [2]:
# ============================================================
# CELL 2: Frozen Backbones, Head, Feature Extraction & Wk 3 Naive Mask
# ============================================================
def load_backbone(name: str):
    if name in ("dinov2_vits14_reg", "dinov2_vits14"):
        model = torch.hub.load("facebookresearch/dinov2", name)
    elif name == "dino_vits8":
        model = torch.hub.load("facebookresearch/dino:main", "dino_vits8")
    else:
        raise ValueError(name)
    model.eval().to(DEVICE)
    for p in model.parameters():
        p.requires_grad = False
    return model

def preprocess_batch(imgs: torch.Tensor, size: int) -> torch.Tensor:
    x = F.interpolate(imgs, size=(size, size), mode="bilinear", align_corners=False)
    return (x - IMAGENET_MEAN.to(x.device)) / IMAGENET_STD.to(x.device)

@torch.no_grad()
def extract_patch_tokens(model, backbone_name: str, imgs: torch.Tensor, size: int, n_layers: int):
    x = preprocess_batch(imgs, size)
    with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
        if "dinov2" in backbone_name:
            feats = model.get_intermediate_layers(x, n=n_layers, reshape=True)
            return torch.cat(feats, dim=1).float()
        outs = model.get_intermediate_layers(x, n=n_layers)
        grid = size // 8
        maps = [tok[:, 1:, :].transpose(1, 2).reshape(tok.shape[0], tok.shape[2], grid, grid) for tok in outs]
        return torch.cat(maps, dim=1).float()

class LinearSegHead(nn.Module):
    def __init__(self, in_channels: int, num_classes: int = 3):
        super().__init__()
        self.bn = nn.BatchNorm2d(in_channels)
        self.conv = nn.Conv2d(in_channels, num_classes, kernel_size=1)

    def forward(self, feat: torch.Tensor, out_size=(160, 160)):
        return F.interpolate(self.conv(self.bn(feat)), size=out_size, mode="bilinear", align_corners=False)

@torch.no_grad()
def week3_naive_attention_mask(dino_s8, imgs: torch.Tensor, threshold: float = 0.6) -> np.ndarray:
    x = preprocess_batch(imgs.to(DEVICE), 224)
    cls_attn = dino_s8.get_last_selfattention(x)[:, :, 0, 1:]
    B = cls_attn.shape[0]
    upsampled = F.interpolate(cls_attn.reshape(B, 6, 28, 28), size=(160, 160), mode="nearest")

    out = []
    for b in range(B):
        head_masks = []
        for h in range(6):
            flat = upsampled[b, h].flatten()
            vals, idxs = torch.sort(flat, descending=True)
            cum = torch.cumsum(vals, dim=0)
            cut = torch.searchsorted(cum, threshold * cum[-1])
            m = torch.zeros_like(flat, dtype=torch.uint8)
            m[idxs[: cut + 1]] = 1
            head_masks.append(m.reshape(160, 160))
        out.append(torch.stack(head_masks).max(dim=0)[0].cpu().numpy())
    return np.stack(out)

In [3]:
# ============================================================
# CELL 3: Metrics & Training Function (FP16 + Feature Caching)
# ============================================================
def update_confusion(cm: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray, num_classes: int = 3):
    valid = (y_true >= 0) & (y_true < num_classes)
    hist = np.bincount(
        num_classes * y_true[valid].astype(np.int64) + y_pred[valid].astype(np.int64),
        minlength=num_classes ** 2,
    ).reshape(num_classes, num_classes)
    return cm + hist

def iou_from_cm(cm: np.ndarray):
    inter = np.diag(cm)
    union = cm.sum(axis=1) + cm.sum(axis=0) - inter
    ious = np.where(union > 0, inter / np.maximum(union, 1), 0.0)
    return {
        "bg_iou": float(ious[0]),
        "target_iou": float(ious[1]),
        "other_iou": float(ious[2]) if len(ious) > 2 else 0.0,
        "miou": float(ious.mean()),
    }

def train_setup(name, backbone_name, img_size, n_layers, epochs=20, lr=1e-3, cache_features=False):
    print(f"\n=== Training {name} ({backbone_name}, {img_size}px, n={n_layers}) ===")
    model = load_backbone(backbone_name)
    head = LinearSegHead(384 * n_layers, 3).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(weight=class_weights)

    bs = 32 if img_size == 224 else 16
    tr_loader = DataLoader(DESISegDataset(train_records), batch_size=bs, shuffle=True, num_workers=2)
    va_loader = DataLoader(DESISegDataset(val_records), batch_size=bs, shuffle=False, num_workers=2)

    tr_cache, va_cache = None, None
    if cache_features:
        tr_cache = [
            (extract_patch_tokens(model, backbone_name, imgs.to(DEVICE), img_size, n_layers).half().cpu(), masks)
            for _, imgs, masks in tqdm(tr_loader, desc="Caching train", leave=False)
        ]
        va_cache = [
            (extract_patch_tokens(model, backbone_name, imgs.to(DEVICE), img_size, n_layers).half().cpu(), masks)
            for _, imgs, masks in tqdm(va_loader, desc="Caching val", leave=False)
        ]

    best_miou, history = -1.0, []
    for ep in range(1, epochs + 1):
        head.train()
        loss_sum = 0.0
        tr_iter = tr_cache if cache_features else tr_loader
        for batch in tr_iter:
            if cache_features:
                feats, masks = batch[0].float().to(DEVICE), batch[1].to(DEVICE)
            else:
                _, imgs, masks = batch
                feats = extract_patch_tokens(model, backbone_name, imgs.to(DEVICE), img_size, n_layers)
                masks = masks.to(DEVICE)

            opt.zero_grad()
            loss = crit(head(feats), masks)
            loss.backward()
            opt.step()
            loss_sum += loss.item()

        head.eval()
        val_cm = np.zeros((3, 3), dtype=np.int64)
        va_iter = va_cache if cache_features else va_loader
        with torch.no_grad():
            for batch in va_iter:
                if cache_features:
                    feats, masks = batch[0].float().to(DEVICE), batch[1]
                else:
                    _, imgs, masks = batch
                    feats = extract_patch_tokens(model, backbone_name, imgs.to(DEVICE), img_size, n_layers)
                preds = head(feats).argmax(dim=1).cpu().numpy()
                val_cm = update_confusion(val_cm, masks.numpy(), preds)

        st = iou_from_cm(val_cm)
        history.append({"epoch": ep, "loss": loss_sum / len(tr_iter), **st})
        if st["miou"] > best_miou:
            best_miou = st["miou"]
            torch.save(
                {"state_dict": head.state_dict(), "backbone": backbone_name, "img_size": img_size, "n_layers": n_layers},
                CKPT_DIR / f"{name}.pt",
            )

        if ep in (1, 5, 10, 15, epochs):
            print(f"Ep {ep:02d} | loss={history[-1]['loss']:.4f} | val mIoU={st['miou']:.3f} | target IoU={st['target_iou']:.3f}")

    pd.DataFrame(history).to_csv(METRIC_DIR / f"{name}_history.csv", index=False)
    del model
    torch.cuda.empty_cache()

In [4]:
# ============================================================
# CELL 4: Train All 4 Setups (Tasks C, D & Ablations)
# ============================================================
SETUPS = [
    ("linear_224",  "dinov2_vits14_reg", 224, 1, True),
    ("linear_448",  "dinov2_vits14_reg", 448, 1, False),
    ("dino_s8_224", "dino_vits8",        224, 1, True),
    ("ms_448",      "dinov2_vits14_reg", 448, 4, False),
]

for name, bname, sz, nl, cache in SETUPS:
    train_setup(name, bname, sz, nl, epochs=20, lr=1e-3, cache_features=cache)


=== Training linear_224 (dinov2_vits14_reg, 224px, n=1) ===
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_reg4_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_reg4_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 202MB/s]


Caching train:   0%|          | 0/66 [00:00<?, ?it/s]

Caching val:   0%|          | 0/10 [00:00<?, ?it/s]

... (14 outputs left out)

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b1d254cccc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b1d254cccc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Ep 15 | loss=0.0850 | val mIoU=0.761 | target IoU=0.666


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b1d254cccc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b1d254cccc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Ep 20 | loss=0.0788 | val mIoU=0.773 | target IoU=0.684


In [5]:
# ============================================================
# CELL 5: Test-Set Evaluation (with TTA) & Comparison Table (Task E)
# ============================================================
@torch.no_grad()
def predict_batch(head, model, backbone_name, imgs, img_size, n_layers, use_tta=False):
    imgs = imgs.to(DEVICE)
    if not use_tta:
        return head(extract_patch_tokens(model, backbone_name, imgs, img_size, n_layers)).argmax(dim=1).cpu().numpy()

    probs_sum = None
    for s in (336, 448, 560):
        for flip in (False, True):
            inp = torch.flip(imgs, dims=[3]) if flip else imgs
            probs = F.softmax(head(extract_patch_tokens(model, backbone_name, inp, s, n_layers)), dim=1)
            if flip:
                probs = torch.flip(probs, dims=[3])
            probs_sum = probs if probs_sum is None else (probs_sum + probs)
    return (probs_sum / 6.0).argmax(dim=1).cpu().numpy()

dinov2 = load_backbone("dinov2_vits14_reg")
dino_s8 = load_backbone("dino_vits8")
backbones = {"dinov2_vits14_reg": dinov2, "dino_vits8": dino_s8}

heads = {}
for name, _, _, _, _ in SETUPS:
    ckpt = torch.load(CKPT_DIR / f"{name}.pt", map_location=DEVICE)
    h = LinearSegHead(384 * ckpt["n_layers"], 3).to(DEVICE)
    h.load_state_dict(ckpt["state_dict"])
    h.eval()
    heads[name] = (h, ckpt["backbone"], ckpt["img_size"], ckpt["n_layers"])

test_loader = DataLoader(DESISegDataset(test_records), batch_size=8, shuffle=False, num_workers=2)
cms = {k: np.zeros((3, 3), dtype=np.int64) for k in ["linear_224", "linear_448", "dino_s8_224", "ms_no_tta", "ms_tta"]}
naive_target_cm = np.zeros((2, 2), dtype=np.int64)
naive_all_cm    = np.zeros((2, 2), dtype=np.int64)
gallery = {}

for keys, imgs, masks in tqdm(test_loader, desc="Test Evaluation"):
    y_true = masks.numpy()
    naive = week3_naive_attention_mask(dino_s8, imgs)
    naive_target_cm = update_confusion(naive_target_cm, (y_true == 1).astype(np.int64), naive, 2)
    naive_all_cm    = update_confusion(naive_all_cm,    (y_true > 0).astype(np.int64),  naive, 2)

    preds = {}
    for name in ["linear_224", "linear_448", "dino_s8_224"]:
        h, bname, sz, nl = heads[name]
        preds[name] = predict_batch(h, backbones[bname], bname, imgs, sz, nl, use_tta=False)
        cms[name] = update_confusion(cms[name], y_true, preds[name])

    h, bname, sz, nl = heads["ms_448"]
    preds["ms_no_tta"] = predict_batch(h, backbones[bname], bname, imgs, sz, nl, use_tta=False)
    preds["ms_tta"]    = predict_batch(h, backbones[bname], bname, imgs, sz, nl, use_tta=True)
    cms["ms_no_tta"] = update_confusion(cms["ms_no_tta"], y_true, preds["ms_no_tta"])
    cms["ms_tta"]    = update_confusion(cms["ms_tta"],    y_true, preds["ms_tta"])

    for i, key in enumerate(keys):
        if key in hard_ids:
            gallery[key] = (imgs[i].permute(1, 2, 0).numpy(), y_true[i], naive[i], preds["linear_224"][i], preds["ms_tta"][i])

rows = [{
    "Setup": "Naive attention (Wk 3)", "Backbone": "DINO ViT-S/8", "Input": 224, "Grid": "28x28",
    "mIoU": np.nan,
    "Target IoU": round(iou_from_cm(naive_target_cm)["target_iou"], 4),
    "All-Sources IoU": round(iou_from_cm(naive_all_cm)["target_iou"], 4),
}]

meta = {
    "linear_224":  ("Linear",       "DINOv2 ViT-S/14+reg", 224, "16x16"),
    "linear_448":  ("Linear",       "DINOv2 ViT-S/14+reg", 448, "32x32"),
    "dino_s8_224": ("Linear",       "DINO ViT-S/8",        224, "28x28"),
    "ms_no_tta":   ("+ms (no TTA)", "DINOv2 ViT-S/14+reg", 448, "32x32"),
    "ms_tta":      ("+ms (+TTA)",   "DINOv2 ViT-S/14+reg", 448, "32x32"),
}

for k, (setup_lbl, b_lbl, inp, grid) in meta.items():
    s = iou_from_cm(cms[k])
    rows.append({
        "Setup": setup_lbl, "Backbone": b_lbl, "Input": inp, "Grid": grid,
        "mIoU": round(s["miou"], 4),
        "Target IoU": round(s["target_iou"], 4),
        "All-Sources IoU": np.nan,
    })

comp_df = pd.DataFrame(rows)
comp_df.to_csv(METRIC_DIR / "comparison.csv", index=False)
display(comp_df)

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main


Test Evaluation:   0%|          | 0/75 [00:00<?, ?it/s]

,Setup,Backbone,Input,Grid,mIoU,Target IoU,All-Sources IoU
0,Naive attention (Wk 3),DINO ViT-S/8,224,28x28,NaN,0.0383,0.1451
1,Linear,DINOv2 ViT-S/14+reg,224,16x16,0.6136,0.4795,NaN
2,Linear,DINOv2 ViT-S/14+reg,448,32x32,0.7273,0.5995,NaN
3,Linear,DINO ViT-S/8,224,28x28,0.7498,0.6482,NaN
4,+ms (no TTA),DINOv2 ViT-S/14+reg,448,32x32,0.7902,0.6994,NaN
5,+ms (+TTA),DINOv2 ViT-S/14+reg,448,32x32,0.8105,0.7292,NaN


In [6]:
# ============================================================
# CELL 6: Hard-Set Gallery & Sync Results Back to Google Drive
# ============================================================
cmap = ListedColormap(["black", "gold", "royalblue"])
for start in range(0, len(hard_ids), 5):
    chunk = [k for k in hard_ids[start : start + 5] if k in gallery]
    fig, axes = plt.subplots(len(chunk), 5, figsize=(14, 2.8 * len(chunk)), squeeze=False)
    for r, key in enumerate(chunk):
        img, gt, naive, lin, ms = gallery[key]
        for c, (panel, title) in enumerate(zip([img, gt, naive, lin, ms], [key, "Pseudo-label", "Naive (Wk 3)", "Linear 224", "+ms (+TTA)"])):
            if c == 0:
                axes[r, c].imshow(np.clip(panel, 0, 1))
            elif c == 2:
                axes[r, c].imshow(panel, cmap="gray", vmin=0, vmax=1)
            else:
                axes[r, c].imshow(panel, cmap=cmap, vmin=0, vmax=2)
            axes[r, c].set_title(title, fontsize=9)
            axes[r, c].axis("off")
    fig.tight_layout()
    fig.savefig(PRED_DIR / f"hard_set_{start // 5 + 1}.png", dpi=130)
    plt.show()

# Sync checkpoints, metrics, and predictions back to Google Drive
!cp -r /content/Week-4/results/checkpoints /content/drive/MyDrive/Week-4/results/
!cp -r /content/Week-4/results/metrics /content/drive/MyDrive/Week-4/results/
!cp -r /content/Week-4/results/predictions /content/drive/MyDrive/Week-4/results/
print("All results synced back to Google Drive!")

Output hidden; open in https://colab.research.google.com to view.